# Preparação dos Dados

- **Projeto:** Recuperação da Confiabilidade dos Dados de Vendas — Café Chain
- **Etapa:** Conexão com o Banco de Dados
- **Dataset:** [Cafe Sales - Dirty Data for Cleaning Training](https://www.kaggle.com/datasets/ahmedmohamed2003/cafe-sales-dirty-data-for-cleaning-training) (Kaggle, CC BY-SA 4.0)
- **Notebook anterior:** [`02_connection_postgres.ipynb`](02_connection_postgres.ipynb)

Este notebook realiza a **carga** da base de dados limpas de vendas para as análises e respostas das perguntas de negócios com SQL.

**Sumário**

1. Carga da base limpa
2. Carregando as variáveis de ambiente (`.env`)
3. Enviando a base de dados para o `PostgreSQL`
4. Verificações dos dados dentro do banco

In [1]:
import os

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine

## 1. Carga da base limpa

In [2]:
df = pd.read_parquet(r'../data/processed/clean_cafe_sales.parquet')
df.head()

,transaction_id,item,quantity,price_per_unit,total_spent,payment_method,location,transaction_date,month,quarter,weekday
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08,9,3,Friday
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16,5,2,Tuesday
2,TXN_4271903,Cookie,4,1.0,4.0,Credit Card,In-store,2023-07-19,7,3,Wednesday
3,TXN_7034554,Salad,2,5.0,10.0,Not Specified,Not Specified,2023-04-27,4,2,Thursday
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11,6,2,Sunday


## 2. Carregando as variáveis de ambiente (`.env`)

Etapa fundamental para o `SQLAlchemy` enviar as credenciais do PostgreSQL para o banco de dados.

In [3]:
load_dotenv()

username = os.getenv("DB_USERNAME")
password = os.getenv("DB_PASSWORD")
host = os.getenv("DB_HOST")
port = os.getenv("DB_PORT")
database = os.getenv("DB_DATABASE")

## 3. Enviando a base de dados para o `PostgreSQL`

In [4]:
engine = create_engine(f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}")

try:
    table_name = 'sales'
    df.to_sql(table_name, con=engine, if_exists='replace', index=False)

    print(f"Os dados foram carregados com sucesso na tabela '{table_name}' no banco de dados de dados '{database}'.")

except Exception as e:
    print(f"Erro ao carregar os dados: {e}")

Os dados foram carregados com sucesso na tabela 'sales' no banco de dados de dados 'cafe_sales'.


## 4. Verificações dos dados dentro do banco

### 4.1 Total de linhas

In [5]:
query_count = "SELECT COUNT(*) AS total_lines FROM sales;"
print(f"Total de linhas no banco: {pd.read_sql(query_count, engine)['total_lines'][0]}")

Total de linhas no banco: 9974


### 4.2 Primeiras linhas da tabela 

In [6]:
query_preview = "SELECT * FROM sales LIMIT 5;"
pd.read_sql(query_preview, engine)

,transaction_id,item,quantity,price_per_unit,total_spent,payment_method,location,transaction_date,month,quarter,weekday
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08,9,3,Friday
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16,5,2,Tuesday
2,TXN_4271903,Cookie,4,1.0,4.0,Credit Card,In-store,2023-07-19,7,3,Wednesday
3,TXN_7034554,Salad,2,5.0,10.0,Not Specified,Not Specified,2023-04-27,4,2,Thursday
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11,6,2,Sunday
